# Reproduce the results

Reproduces the results of *A Quasi-Experimental Evaluation of Rail Station Effects on Property Values in Devon* (MSc dissertation, University of Exeter, 2026) from the code in this repository and the data on Zenodo ([10.5281/zenodo.24544649](https://doi.org/10.5281/zenodo.24544649)).

The data files are restricted. Request access on the Zenodo record, then download the five zip files into one folder:

- **Google Colab:** put the zips in a Google Drive folder named `zenodo_data`. If they are in a different Drive folder, type its name in the `DATA` box in step 1. [Open this notebook in Colab](https://colab.research.google.com/github/Opeolumike/Rail-Station-Effects-on-Property-Values/blob/main/reproduce.ipynb).
- **Own computer:** clone the repository, put the zips in a folder named `zenodo_data` inside it (ignored by git), or set `DATA` in step 1 to the full path of another folder (with forward slashes, for example `C:/Users/me/zips`), and open this notebook from the repository folder in Jupyter or VS Code.

Then run the steps in order. Step 3 takes several hours, mainly notebook 04.

## 1. Set up
Finds the repository, unzips the data into it and checks every file against `MANIFEST-SHA256.txt`.

In [ ]:
import os, sys, re, json, glob, shutil, hashlib, zipfile, subprocess
import pandas as pd

# Folder with the five zips: a folder name inside My Drive (Google Colab) or inside the repository (own computer), or a full path.
# On Windows, write a full path with forward slashes, for example 'C:/Users/me/zips'.
DATA = 'zenodo_data'  #@param {type:"string"}

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    R = '/content/Rail-Station-Effects-on-Property-Values'
    DATA = os.path.join('/content/drive/MyDrive', DATA)
    if not os.path.isdir(R):
        subprocess.run(['git', 'clone', 'https://github.com/Opeolumike/Rail-Station-Effects-on-Property-Values.git', R], check=True)
else:
    R = os.path.abspath('.')                   # this notebook sits in the repository folder
    DATA = os.path.join(R, DATA)

zips = sorted(glob.glob(os.path.join(DATA, '*.zip')))
assert zips, f'No zip files found in {DATA}'
for z in zips:
    print('Unzipping', os.path.basename(z))
    with zipfile.ZipFile(z) as zf:
        zf.extractall(R)

bad = []
with open(os.path.join(R, 'MANIFEST-SHA256.txt')) as manifest:
    for line in manifest:
        expected, path = line.rstrip('\n').split('  ', 1)
        f = os.path.join(R, path)
        if not os.path.isfile(f):
            bad.append(path + ' (missing)')
            continue
        h = hashlib.sha256()
        with open(f, 'rb') as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b''):
                h.update(chunk)
        if h.hexdigest() != expected:
            bad.append(path)
print('All files match the manifest' if not bad else 'Files that do not match:\n' + '\n'.join(bad))

## 2. Keep the original results
The notebooks overwrite `04_results` and `05_logs`, so the original tables and attrition log are copied aside first.

In [ ]:
ORIG = os.path.join(DATA, 'original_results')
shutil.copytree(os.path.join(R, '04_results', 'tables'), os.path.join(ORIG, 'tables'), dirs_exist_ok=True)
shutil.copy(os.path.join(R, '05_logs', 'script_logs', 'attrition_log.json'), ORIG)
print('Original results saved to', ORIG)

## 3. Run notebooks 01 to 06
Runs each notebook in order without changing it. Executed copies are saved to `DATA/executed`. To resume after an interruption, set `START` to the first notebook that did not finish.

In [ ]:
notebooks = ['01_clean_tabular_data', '02_spatial_bridge', '03_geometric_foundation',
             '04_network_analysis', '05_econometric_regression', '06_final_layers']
START = '01_clean_tabular_data'

completed = True
for nb in notebooks[notebooks.index(START):]:
    print('Running', nb, '...')
    run = subprocess.run([sys.executable, '-m', 'jupyter', 'nbconvert', '--to', 'notebook', '--execute', f'{nb}.ipynb',
                          '--ExecutePreprocessor.timeout=-1', '--output-dir', os.path.join(DATA, 'executed')],
                         cwd=os.path.join(R, '03_scripts'), capture_output=True, text=True)
    if run.returncode != 0:
        completed = False
        print('Stopped at', nb)
        print(run.stderr[-3000:])
        break
    print('Done', nb)

## 4. Compare with the original results
Every table is compared with the original at a relative tolerance of 1e-6. Two differences are expected and explained:

- **z statistics of non-identified terms.** In the urban model, Treated x Near and Treated x Near x Post cannot be estimated (no Marsh Barton near-zone sales), so their coefficients are effectively zero and their z statistics are rounding noise. These rows are excluded from the z statistic check.
- **Attrition log.** The log adds a timestamped entry each time a step runs, so each rerun entry is checked against all original entries of the same step. The walk network counts differ because the original log recorded an earlier download (37,362 nodes). The locked network used for all routing has 37,221 nodes. The original log also holds a `Note` under `Phase_2A_EPC_Bridge` that no notebook writes, so that step shows as different too.

In [ ]:
assert completed, 'Step 3 did not finish, so the comparison would not be valid.'

def compare_tables(a, b):
    try:
        pd.testing.assert_frame_equal(a, b, check_exact=False, rtol=1e-6)
        return 'MATCH'
    except AssertionError:
        if {'Coefficient', 'z_Statistic'} <= set(a.columns):
            identified = a['Coefficient'].abs() > 1e-10
            a2 = a.copy()
            b2 = b.copy()
            a2.loc[~identified, 'z_Statistic'] = 0
            b2.loc[~identified, 'z_Statistic'] = 0
            try:
                pd.testing.assert_frame_equal(a2, b2, check_exact=False, rtol=1e-6)
                return 'MATCH (z statistics of non-identified terms excluded)'
            except AssertionError:
                pass
        return 'DIFFERS'

for f in sorted(glob.glob(os.path.join(ORIG, 'tables', '*.csv'))):
    name = os.path.basename(f)
    print(f'{compare_tables(pd.read_csv(f), pd.read_csv(os.path.join(R, "04_results", "tables", name))):55} {name}')

stamp = re.compile(r'_(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}|\d{8}_\d{4})$')
def by_step(log):
    steps = {}
    for key, value in log.items():
        steps.setdefault(stamp.sub('', key), []).append(value)
    return steps

original = by_step(json.load(open(os.path.join(ORIG, 'attrition_log.json'))))
rerun = by_step(json.load(open(os.path.join(R, '05_logs', 'script_logs', 'attrition_log.json'))))
print()
for step in sorted(set(original) | set(rerun)):
    if all(v in original.get(step, []) for v in rerun.get(step, [])) and step in rerun:
        print(f'{"MATCH":55} attrition log: {step}')
    else:
        print(f'{"DIFFERS":55} attrition log: {step}')
        print('    original:', original.get(step))
        print('    rerun:   ', rerun.get(step))

pd.read_csv(os.path.join(R, '04_results', 'tables', 'headline_results.csv'))

## 5. Record the versions used

In [ ]:
from importlib.metadata import version, PackageNotFoundError
print('Python', sys.version.split()[0])
for p in ['pandas', 'numpy', 'statsmodels', 'scipy', 'geopandas', 'shapely', 'pyproj', 'fiona', 'pyarrow',
          'fastparquet', 'osmnx', 'networkx', 'matplotlib', 'contextily', 'tqdm']:
    try:
        print(f'{p}=={version(p)}')
    except PackageNotFoundError:
        print(f'{p} not installed')